In [ ]:
# Copyright (C) 2025-2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

# EfficientSAM3 Demo

EfficientSAM3 is a distilled version of SAM3 that uses lightweight student backbones
(EfficientViT, RepViT, TinyViT) to achieve faster inference while trading off some
segmentation quality.

EfficientSAM3 **inherits the full SAM3 high-level API** — text prompts, box prompts,
point prompts, and visual exemplar mode all work via the same `Sample`-based interface.

This notebook demonstrates:

1. **Text prompts** — zero-shot detection by category name
2. **Box prompts** — segment within bounding boxes
3. **Point prompts** — click-to-segment
4. **Visual exemplar mode** — fit on a reference image, predict on new images

**Available backbones**: EfficientViT-B0/B1/B2, RepViT-M0.9/M1.1/M2.3, TinyViT-5M/11M/21M

> **Note**: EfficientSAM3 may produce lower confidence scores than SAM3 due to the
> distillation process. The `presence_token_head` is disabled by default for robustness
> across checkpoints (see [Appendix](#appendix-presence-token-analysis) for details).

Images from the [EfficientSAM3](https://github.com/SimonZeng7108/efficientsam3)
repository and COCO dataset.

## Setup

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from instantlearn.data.base import Category, Sample
from instantlearn.data.base.prediction import Prediction
from instantlearn.device import resolve_device_for_runtime
from instantlearn.models import EfficientSAM3
from instantlearn.utils.constants import Backend

In [ ]:
# Select a physical device and resolve its PyTorch runtime ID
device_info, runtime_device = resolve_device_for_runtime(
    device=None,
    runtime=Backend.TORCH,
    supported_device_types=EfficientSAM3.card().supported_device_types(Backend.TORCH),
)
device = torch.device(runtime_device)

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()  # noqa: PLC2801
    if torch.cuda.get_device_properties(device).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

print(f"Using physical device: {device_info.name} ({device})")  # noqa: T201

### Visualization helpers

In [ ]:
def show_mask(
    mask: np.ndarray,
    ax: plt.Axes,
    random_color: bool = False,
    borders: bool = True,
) -> None:
    """Overlay a mask on the given axes."""
    if random_color:
        rng = np.random.default_rng()
        color = np.concatenate([rng.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2)
    ax.imshow(mask_image)


def show_points(
    coords: np.ndarray,
    ax: plt.Axes,
    marker_size: int = 375,
) -> None:
    """Display point prompts (green markers)."""
    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        color="green",
        marker="*",
        s=marker_size,
        edgecolor="white",
        linewidth=1.25,
    )


def show_box(box: np.ndarray, ax: plt.Axes) -> None:
    """Display a bounding box in XYXY format."""
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(
        plt.Rectangle((x0, y0), w, h, edgecolor="green", facecolor=(0, 0, 0, 0), lw=2),
    )


def show_predictions(
    image: np.ndarray | Image.Image,
    prediction: Prediction,
    categories: list[str] | None = None,
    title: str | None = None,
) -> None:
    """Display prediction masks, boxes, and labels.

    Args:
        image: Input image (numpy HWC or PIL).
        prediction: Model prediction with masks, boxes, scores, and label ids.
        categories: Category names for labeling.
        title: Plot title.
    """
    masks = prediction.masks
    boxes = prediction.boxes if prediction.boxes is not None else np.empty((0, 4), dtype=np.float32)
    labels = prediction.label_ids

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    for mask in masks:
        show_mask(mask, plt.gca(), random_color=True)
    for i, box in enumerate(boxes):
        score = prediction.scores[i] if i < len(prediction.scores) else 0
        label_id = int(labels[i])
        label_text = categories[label_id] if categories and label_id < len(categories) else str(label_id)
        show_box(box[:4], plt.gca())
        plt.gca().text(
            box[0],
            box[1] - 4,
            f"{label_text} {score:.2f}",
            fontsize=12,
            color="white",
            bbox={"facecolor": "green", "alpha": 0.7, "pad": 2},
        )
    if title:
        plt.title(title, fontsize=18)
    plt.axis("off")
    plt.show()

## Load model and images

In [ ]:
model = EfficientSAM3(device=device_info)
print(f"EfficientSAM3 loaded on {model.device_info.name} ({model.device})!")  # noqa: T201

In [ ]:
# Load example images
truck_pil = Image.open("assets/efficientsam3/truck.jpg")
court_pil = Image.open("assets/efficientsam3/basketball_court.jpg")

# Keep HWC numpy arrays for Sample; CHW tensors are used only in the debug appendix.
truck_image = np.array(truck_pil)
court_image = np.array(court_pil)
truck_tensor = torch.from_numpy(truck_image).permute(2, 0, 1).float()
court_tensor = torch.from_numpy(court_image).permute(2, 0, 1).float()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(truck_pil)
axes[0].set_title("truck.jpg")
axes[0].axis("off")
axes[1].imshow(court_pil)
axes[1].set_title("basketball_court.jpg")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## 1. Text Prompts

Provide category names and EfficientSAM3 detects all matching objects.
No `fit()` call needed — categories are specified per sample.

### 1a. Single category

In [ ]:
sample = Sample(image=court_image, categories=[Category(0, "shoe")])
predictions = model.predict(sample)
pred = predictions[0]

print(f"Text prompt: 'shoe' -> {len(pred.masks)} detection(s)")  # noqa: T201
show_predictions(court_pil, pred, categories=["shoe"], title="Text Prompt: 'shoe'")

### 1b. Multiple categories

In [ ]:
category_names = ["person", "shoe"]
sample = Sample(image=court_image, categories=[Category(i, name) for i, name in enumerate(category_names)])
predictions = model.predict(sample)
pred = predictions[0]

print(f"Text prompts: {category_names} -> {len(pred.masks)} detection(s)")  # noqa: T201
show_predictions(court_pil, pred, categories=category_names, title=f"Text Prompts: {category_names}")

---
## 2. Box Prompts

Provide bounding boxes in `[x1, y1, x2, y2]` format to segment specific regions.

In [ ]:
input_box = np.array([[425, 600, 700, 875]])

sample = Sample(image=truck_image, bboxes=input_box)
predictions = model.predict(sample)
pred = predictions[0]

print(f"Box prompt -> {len(pred.masks)} detection(s)")  # noqa: T201
show_predictions(truck_pil, pred, title="Box Prompt: [425, 600, 700, 875]")

---
## 3. Point Prompts

Click-to-segment: provide `(x, y)` pixel coordinates to identify objects.


In [ ]:
input_points = np.array([[520, 375]])

sample = Sample(image=truck_image, points=input_points)
predictions = model.predict(sample)
pred = predictions[0]

print(f"Point prompt -> {len(pred.masks)} detection(s)")  # noqa: T201

# Visualize with point overlay
masks_np = pred.masks
plt.figure(figsize=(10, 10))
plt.imshow(truck_pil)
for mask in masks_np:
    show_mask(mask, plt.gca(), random_color=True)
show_points(input_points, plt.gca())
plt.title("Point Prompt: (520, 375)", fontsize=18)
plt.axis("off")
plt.show()

---
## 4. Visual Exemplar Mode

Exemplar mode caches geometry features from a reference image + prompt and
reuses them to detect similar objects in new images — no text, box, or point
prompts needed on targets.

Workflow:
1. Create a model with `prompt_mode="visual_exemplar"`
2. Construct a reference `Sample` with `image`, `bboxes` (or `points`), and `categories=[Category(...)]`
3. Call `model.fit(reference)` to encode and cache the reference prompts
4. Call `model.predict(target)` on new images — detections are driven by
   the cached exemplar features

### Load elephant images

In [ ]:
coco_dir = "assets/coco"

ref_pil = Image.open(f"{coco_dir}/000000286874.jpg")  # Reference elephant
target1_pil = Image.open(f"{coco_dir}/000000390341.jpg")  # Target 1
target2_pil = Image.open(f"{coco_dir}/000000173279.jpg")  # Target 2

ref_image = np.array(ref_pil)
target1_image = np.array(target1_pil)
target2_image = np.array(target2_pil)
ref_tensor = torch.from_numpy(ref_image).permute(2, 0, 1).float()
target1_tensor = torch.from_numpy(target1_image).permute(2, 0, 1).float()
target2_tensor = torch.from_numpy(target2_image).permute(2, 0, 1).float()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(ref_pil)
axes[0].set_title("Reference")
axes[1].imshow(target1_pil)
axes[1].set_title("Target 1")
axes[2].imshow(target2_pil)
axes[2].set_title("Target 2")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 4a. Single-shot box exemplar

Provide a bounding box around the elephant in the reference image. The model
encodes the region's geometry features and uses them to detect elephants in
new images.

In [ ]:
model_ve = EfficientSAM3(
    device=device_info,
    prompt_mode="visual_exemplar",
)

# Reference: one elephant with a bounding box
ref_sample = Sample(
    image=ref_image,
    bboxes=np.array([[132, 133, 458, 405]]),  # xyxy around the elephant
    categories=[Category(0, "elephant")],
)

# Fit on the reference - encodes and caches geometry features
model_ve.fit(ref_sample)
print("Reference encoded! Ready to predict on new images.")  # noqa: T201

# Show the reference with the bounding box
plt.figure(figsize=(8, 6))
plt.imshow(ref_pil)
show_box(np.array([132, 133, 458, 405]), plt.gca())
plt.title("Reference: elephant bbox", fontsize=16)
plt.axis("off")
plt.show()

In [ ]:
# Predict on target images — no prompts needed
targets = [
    Sample(image=target1_image),
    Sample(image=target2_image),
]
predictions = model_ve.predict(targets)

target_images = [target1_pil, target2_pil]
for i, (img, pred) in enumerate(zip(target_images, predictions, strict=False)):
    n_det = len(pred.masks)
    print(f"Target {i + 1}: {n_det} detection(s)")  # noqa: T201
    show_predictions(img, pred, categories=["elephant"], title=f"Exemplar Prediction — Target {i + 1}")

### 4b. Single-shot point exemplar

Instead of a bounding box, a single point click on the reference object is
enough to detect similar objects in new images.

In [ ]:
model_pt = EfficientSAM3(
    device=device_info,
    prompt_mode="visual_exemplar",
)

# Reference: point at the elephant's center
ref_sample_pt = Sample(
    image=ref_image,
    points=np.array([[295, 270]]),  # center of the elephant
    categories=[Category(0, "elephant")],
)

model_pt.fit(ref_sample_pt)
print("Reference (point) encoded!")  # noqa: T201

# Predict on the same targets
predictions_pt = model_pt.predict(targets)

for i, (img, pred) in enumerate(zip(target_images, predictions_pt, strict=False)):
    n_det = len(pred.masks)
    print(f"Target {i + 1}: {n_det} detection(s)")  # noqa: T201
    show_predictions(img, pred, categories=["elephant"], title=f"Point Exemplar - Target {i + 1}")

### 4c. N-shot cross-image exemplar

Provide multiple reference images with prompts for the same category. The model
concatenates geometry features across images for stronger exemplar representation.

In [ ]:
model_nshot = EfficientSAM3(
    device=device_info,
    prompt_mode="visual_exemplar",
)

# Two reference images, both showing elephants
ref_samples = [
    Sample(
        image=ref_image,
        points=np.array([[295, 270]]),
        categories=[Category(0, "elephant")],
    ),
    Sample(
        image=target1_image,
        points=np.array([[320, 220]]),  # point on elephant in target1
        categories=[Category(0, "elephant")],
    ),
]

model_nshot.fit(ref_samples)
print("2-shot cross-image reference encoded!")  # noqa: T201

# Predict on a third image
predictions_nshot = model_nshot.predict(Sample(image=target2_image))
pred = predictions_nshot[0]

print(f"Target: {len(pred.masks)} detection(s)")  # noqa: T201
show_predictions(target2_pil, pred, categories=["elephant"], title="N-Shot Cross-Image Exemplar")

---
## Summary

| Prompt Type | API | Input | Description |
|---|---|---|---|
| Text | `model.predict(Sample(..., categories=[...]))` | Category names | Zero-shot detection by name |
| Box | `model.predict(Sample(..., bboxes=...))` | XYXY boxes | Segment specific regions |
| Point | `model.predict(Sample(..., points=...))` | XY coordinates | Click-to-segment (positive only) |
| Visual exemplar (box) | `model.fit(ref)` then `model.predict(target)` | Reference bbox | Detect similar objects in new images |
| Visual exemplar (point) | `model.fit(ref)` then `model.predict(target)` | Reference point | Detect similar objects in new images |
| N-shot cross-image | `model.fit([ref1, ref2])` then `model.predict(target)` | Multiple refs | Stronger exemplar from multiple images |

**Key differences from SAM3:**
- Lower confidence scores due to distillation
- `presence_token_head` disabled by default
- Uses lightweight backbones (EfficientViT, RepViT, TinyViT) for faster inference
- Default backbone: EfficientViT-B2

---
## Appendix: Presence Token Analysis

The `presence_token_head` in the DETR decoder can optionally gate detection scores:
`final_score = pred_logits.sigmoid() × presence.sigmoid()`

In early EfficientSAM3 checkpoints, the presence head produced strongly negative logits
(~-3.78, sigmoid ~0.02), crushing all scores to near-zero
([efficientsam3#17](https://github.com/SimonZeng7108/efficientsam3/issues/17)).
However, this behavior is **checkpoint-dependent** — some models produce healthy
presence probabilities (>0.8) where the gating works as intended.

Our default (`use_presence=False`) disables this head for robustness across all
checkpoints. The cells below inspect the raw outputs so you can evaluate whether
re-enabling presence would help or hurt for your specific model.

See also [presence_token_analysis.md](../src/instantlearn/models/efficient_sam3/presence_token_analysis.md).

In [ ]:
from transformers import CLIPTokenizerFast

from instantlearn.models.efficient_sam3.model import EfficientSam3Model
from instantlearn.models.sam3.processing import Sam3Postprocessor, Sam3Preprocessor

In [ ]:
debug_model = (
    EfficientSam3Model.from_pretrained(
        backbone_type="efficientvit",
        variant="b0",
    )
    .to(device)
    .eval()
)

# Re-enable presence for debugging
debug_model.detr_decoder.use_presence = True

tokenizer = CLIPTokenizerFast.from_pretrained("jetjodh/sam3")
tokenizer.pad_token_id = 0
STUDENT_CONTEXT_LENGTH = 32

debug_preprocessor = Sam3Preprocessor(target_size=1008).to(device)
debug_postprocessor = Sam3Postprocessor(
    target_size=1008,
    threshold=0.01,
    mask_threshold=0.5,
).to(device)

debug_image_tensor = court_tensor
debug_pixel_values, debug_original_sizes = debug_preprocessor(
    debug_image_tensor.unsqueeze(0).to(device),
)

prompts_to_debug = ["shoe", "person"]
print("Raw model outputs -- presence ENABLED\n")  # noqa: T201

for prompt in prompts_to_debug:
    text_inputs = tokenizer(
        [prompt],
        return_tensors="pt",
        padding="max_length",
        max_length=STUDENT_CONTEXT_LENGTH,
        truncation=True,
    )

    with torch.no_grad():
        debug_vision = debug_model.get_vision_features(debug_pixel_values)
        outputs_with_presence = debug_model(
            vision_embeds=debug_vision,
            input_ids=text_inputs.input_ids.to(device),
            attention_mask=text_inputs.attention_mask.to(device),
        )

    pred_logits = outputs_with_presence["pred_logits"]
    presence_logits = outputs_with_presence["presence_logits"]

    class_probs = pred_logits.sigmoid()
    presence_prob = presence_logits.sigmoid() if presence_logits is not None else None

    top_k = 5
    top_class, _ = class_probs.view(-1).topk(top_k)

    print(f"Prompt: '{prompt}'")  # noqa: T201
    if presence_prob is not None:
        print(f"  Presence logit:  {presence_logits.item():.4f}")  # noqa: T201
        print(f"  Presence prob:   {presence_prob.item():.4f}")  # noqa: T201
    print(f"  Top-{top_k} classification probs: {[f'{p:.4f}' for p in top_class.cpu().tolist()]}")  # noqa: T201
    if presence_prob is not None:
        final_top = (class_probs * presence_prob).view(-1).topk(top_k)[0]
        print(f"  Top-{top_k} final (class*presence): {[f'{p:.4f}' for p in final_top.cpu().tolist()]}")  # noqa: T201

    if presence_prob is not None:
        final_probs = (class_probs * presence_prob).view(-1)
        for thresh in [0.5, 0.1, 0.05, 0.01]:
            n_det = (final_probs > thresh).sum().item()
            print(f"  Detections @ {thresh}: {n_det}")  # noqa: T201
    print()  # noqa: T201

# Interpretation
print("If presence_prob > 0.8, the gating has minimal effect and")  # noqa: T201
print("re-enabling use_presence may be safe for this checkpoint.")  # noqa: T201
print("If presence_prob < 0.1, presence crushes all scores — keep it disabled.")  # noqa: T201

del debug_model  # Free memory